In [0]:
%python
dbutils.widgets.removeAll()

In [0]:

create widget text storageName default "adlsmartdata1702ep";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
DROP CATALOG IF EXISTS catalog_au CASCADE;

In [0]:
CREATE CATALOG IF NOT EXISTS catalog_au
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
DROP SCHEMA IF EXISTS catalog_au.raw;
DROP SCHEMA IF EXISTS catalog_au.bronze;
DROP SCHEMA IF EXISTS catalog_au.silver;
DROP SCHEMA IF EXISTS catalog_au.golden;

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

In [0]:
CREATE SCHEMA IF NOT EXISTS catalog_au.raw;
CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_au.silver;
CREATE SCHEMA IF NOT EXISTS catalog_au.golden;

###Tablas Bronze

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.Automobile_data (
symboling integer,
normalized_losses string,
make string,
fuel_type string,
aspiration  string,
num_of_doors string,
body string ,
drive_wheels string,
engine_location string,
wheel_base string,
length string,
width string,
height string,
curb_weight string,
engine_type string,
num_of_cylinders string,
engine_size string,
fuel_system string, 
bore string,
stroke string,
compression_ratio string,
horsepower string,
peak_rpm string,
city_mpg string,
highway_mpg string,
price string

)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/Automobile_data"

###Tablas Silver

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.silver.Automobile_data_transformed (
symboling integer,
normalized_losses string,
make string,
fuel_type string,
aspiration  string,
num_of_doors string,
body string ,
drive_wheels string,
engine_location string,
wheel_base string,
length string,
width string,
height string,
curb_weight string,
engine_type string,
num_of_cylinders string,
engine_size string,
fuel_system string, 
bore string,
stroke string,
compression_ratio string,
horsepower string,
peak_rpm string,
city_mpg string,
highway_mpg string,
price string

)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/Automobile_data_transformed"

###Tablas Golden

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.golden.Automobile_data_partitioned (
body string ,
wheel_base string
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/Automobile_data_partitioned"